# ac data

In [ ]:
import pandas as pd
import re
import os
from striprtf.striprtf import rtf_to_text
from datetime import datetime

def read_rtf(file_path):
    with open(file_path, 'r') as file:
        rtf_content = file.read()
    text = rtf_to_text(rtf_content)

    # get name of disease from file name
    disease = file_path.split('/')[-1].split('_')[1].split('.')[0]

    return text, disease

def get_year(text):
    year_pattern = r'\b(19\d{2}|20\d{2}|25\d{2})\b'
    years = re.findall(year_pattern, text)

    # find year lower than current year
    current_year = datetime.now().year
    years = [year for year in years if int(year) <= current_year]

    return years[0]

def clean_text(text):
    lines = text.split('\n')
    data_lines = [line for line in lines if line.strip()]
    data = []
    for line in data_lines:
        parsed_line = [x.strip() for x in line.split('\t')]
        # if first element of line is empty, remove it
        if not parsed_line[0]:
            parsed_line = parsed_line[1:]
        # if first element of line is Zone, combine it with next element
        if parsed_line[0] == 'Zone':
            parsed_line[1] = 'Zone ' + parsed_line[1]
            parsed_line = parsed_line[1:]
        if parsed_line:
            data.append(parsed_line)

    # remove first and last line
    year = get_year(text)
    data = data[1:-1]
    return data, year

def fill_name(col_names):
    cleaned_names = []

    for i, name in enumerate(col_names):
        name = name.strip()
        if 'Area' in name:
            cleaned_names.append('Areas')
        elif 'Total' in name:
            cleaned_names.append('Total')
        elif '<1' in name or '<1ปี' in name:
            cleaned_names.append('<1')
        elif '65+' in name:
            cleaned_names.append('65+')
        elif 'ไม่ทรา' in name:
            cleaned_names.append('Unknown')
        elif '+' in name:
            current = name.strip('+').strip()
            if i + 1 < len(col_names) and '+' in col_names[i + 1] and not col_names[i + 1].endswith('+'):
                try:
                    next_age = int(re.sub(r'\D', '', col_names[i + 1]))
                    cleaned_names.append(f"{current}-{next_age - 1}")
                except ValueError:
                    cleaned_names.append(f"{current}-{current}")  # if conversion fails, use current as is
            else:
                cleaned_names.append(f"{current}-{current}")  # No next range, so make it a self range
        elif name.endswith('-'):
            current = name.strip('-').strip()
            if i + 1 < len(col_names):
                # Extract the first sequence of digits from the next column's name
                next_age_match = re.search(r'\d+', col_names[i + 1])
                if next_age_match:
                    next_age = int(next_age_match.group())
                    cleaned_names.append(f"{current}-{next_age - 1}")
                else:
                    cleaned_names.append(current)  # Use current if no digits found
            else:
                cleaned_names.append(current)  # Use current as is if no next info
        else:
            cleaned_names.append(name)  # Directly use clear ranges like '10-14'
    
    total_index = cleaned_names.index('Total')
    under_one_index = cleaned_names.index('<1')
    if under_one_index - total_index == 2:
        cleaned_names[total_index + 1] = '0'

    return cleaned_names
    
def text_to_dataframe(data, year, disease):
    
    data = [[re.sub('ปี', '', x) for x in row] for row in data]
    # convert to dataframe
    data = pd.DataFrame(data[1:], columns=data[0])
    data = data.loc[:, data.apply(lambda col: not all(x == "" for x in col))]
    data.columns = fill_name(data.columns)

    # convert to longer format
    df = data.melt(id_vars=['Areas'], var_name='Age', value_name='Cases')
    df['Year'] = year
    df['Disease'] = disease

    return df

In [43]:
# get disease list
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('ac_')]
    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text,name = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, name)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_ac.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}: {e}')
        continue

* Error in STI Total: No objects to concatenate
* Error in DHF Total: No objects to concatenate
* Error in Encephalitis Total: No objects to concatenate
* Error in TB Total: No objects to concatenate
* Error in Measles Total: No objects to concatenate


# mcd data

In [98]:
import pandas as pd
import re
import os
from striprtf.striprtf import rtf_to_text
from datetime import datetime

def read_rtf(file_path):
    with open(file_path, 'r') as file:
        rtf_content = file.read()
    text = rtf_to_text(rtf_content)

    # get name of disease from file name
    disease = file_path.split('/')[-1].split('_')[1].split('.')[0]

    return text, disease

def get_year(text):
    year_pattern = r'\b(19\d{2}|20\d{2}|25\d{2})\b'
    years = re.findall(year_pattern, text)

    # find year lower than current year
    current_year = datetime.now().year
    years = [year for year in years if int(year) <= current_year]

    return years[0]

def clean_text(text):
    lines = text.split('\n')
    data_lines = [line for line in lines if line.strip()]
    data = []
    for line in data_lines:
        parsed_line = [x.strip() for x in line.split('\t')]
        # if first element of line is empty, remove it
        if not parsed_line[0]:
            parsed_line = parsed_line[1:]
        # if first element of line is Zone, combine it with next element
        if parsed_line[0] == 'Zone':
            parsed_line[1] = 'Zone ' + parsed_line[1]
            parsed_line = parsed_line[1:]
        if parsed_line:
            data.append(parsed_line)

    # remove first and last line
    year = get_year(text)
    data = data[1:-1]
    return data, year

def fill_name(col_names):
    cleaned_names = []

    for i, name in enumerate(col_names):
        name = name.strip()
        if 'Area' in name:
            cleaned_names.append('Areas')
        elif 'Total' in name:
            cleaned_names.append('Total')
        else:
            cleaned_names.append(name)

    return cleaned_names
    
def text_to_dataframe(data, year, disease):
    # create headers
    headers = data[0]
    labels = data[1]
    adjusted_headers = ['Areas']
    for i, header in enumerate(headers[1:]):
        adjusted_headers.append(f'{header}_{labels[i*2-2]}')
        adjusted_headers.append(f'{header}_{labels[i*2-1]}')

    # convert to dataframe
    df = pd.DataFrame(data[2:], columns=adjusted_headers)
    df = df.loc[:, df.apply(lambda col: not all(x == "" for x in col))]
    
    # Melt the DataFrame to longer format
    df = df.melt(id_vars=['Areas'], var_name='Period_Type', value_name='Count')
    df[['Month', 'Type']] = df['Period_Type'].str.split('_', expand=True)
    df.drop('Period_Type', axis=1, inplace=True)

    # if Type contains cas, replace it with Cases
    df['Type'] = df['Type'].apply(lambda x: 'Cases' if 'cas' in x.lower() else x)
    df['Type'] = df['Type'].apply(lambda x: 'Deaths' if 'dea' in x.lower() else x)
    
    # Assign static values for 'Year' and 'Disease'
    df['Year'] = year
    df['Disease'] = disease

    # remove including NAN values
    df = df.dropna()

    # remove rows which not number in Count
    df = df[df['Count'].apply(lambda x: x.isnumeric())]
    df = df.astype({'Count': 'int32'})

    return df

In [100]:
file = '../Data/GetData/HFM/mcd_HFM_65.rtf'
text, name = read_rtf(file)
data, year = clean_text(text)
df = text_to_dataframe(data, year, name)
df

,Areas,Count,Month,Type,Year,Disease
0,Total,100164,Total,Cases,2022,HFM
1,ZONE:01,10871,Total,Cases,2022,HFM
2,Chiang Mai,3460,Total,Cases,2022,HFM
3,Chiang Rai,2546,Total,Cases,2022,HFM
4,Lampang,1200,Total,Cases,2022,HFM
...,...,...,...,...,...,...
2517,Songkhla,0,Dec,Deaths,2022,HFM
2518,Trang,0,Dec,Deaths,2022,HFM
2519,Yala,0,Dec,Deaths,2022,HFM
2520,ZONE:13,0,Dec,Deaths,2022,HFM


In [101]:
# get disease list
disease_list = os.listdir('../Data/GetData/')
for disease in disease_list:
    file_list = os.listdir('../Data/GetData/' + disease)
    file_list = [file for file in file_list if file.startswith('mcd_')]
    outcome = []
    try:
        for file in file_list:
            file_path = f'../Data/GetData/{disease}/{file}'
            text,name = read_rtf(file_path)
            data, year = clean_text(text)
            df = text_to_dataframe(data, year, name)
            outcome.append(df)

        outcome = pd.concat(outcome)
        outcome.to_csv(f'../Data/CleanData/{disease}_mcd.csv', index=False)
    except Exception as e:
        print(f'* Error in {disease}: {e}')
        continue

* Error in STI Total: No objects to concatenate
* Error in DHF Total: No objects to concatenate
* Error in Encephalitis Total: No objects to concatenate
* Error in TB Total: No objects to concatenate
* Error in Measles Total: No objects to concatenate
